In [5]:
import cv2
import mediapipe as mp

print('✅ Libraries loaded!')
print(f'OpenCV version: {cv2.__version__}')
print(f'MediaPipe version: {mp.__version__}')

✅ Libraries loaded!
OpenCV version: 5.0.0
MediaPipe version: 0.10.35


In [6]:
import urllib.request

model_url = 'https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task'
model_path = 'hand_landmarker.task'

urllib.request.urlretrieve(model_url, model_path)
print('✅ Model downloaded!')

✅ Model downloaded!


In [7]:
import os
print(os.path.exists('hand_landmarker.task'))
print(f'File size: {os.path.getsize("hand_landmarker.task") / 1024:.1f} KB')

True
File size: 7635.8 KB


In [8]:
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision

# Configure the hand landmark detector
base_options = mp_python.BaseOptions(model_asset_path='hand_landmarker.task')
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    num_hands=1,                     # track one hand only
    min_hand_detection_confidence=0.7,
    running_mode=vision.RunningMode.VIDEO  # we're processing a live video stream
)

detector = vision.HandLandmarker.create_from_options(options)

print('✅ Hand landmark detector ready!')

✅ Hand landmark detector ready!


In [9]:
import time

# Open webcam
cap = cv2.VideoCapture(0)  # 0 = default webcam

if not cap.isOpened():
    print('❌ Could not open webcam')
else:
    print('✅ Webcam opened! Press "q" in the video window to quit.')

frame_timestamp = 0

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break
    
    # Flip horizontally so it acts like a mirror (feels natural)
    frame = cv2.flip(frame, 1)
    
    # Convert BGR (OpenCV's default) to RGB (MediaPipe's expected format)
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
    
    # Run detection
    frame_timestamp += 1
    result = detector.detect_for_video(mp_image, frame_timestamp)
    
    # Draw landmarks if a hand was detected
    if result.hand_landmarks:
        h, w, _ = frame.shape
        for hand_landmarks in result.hand_landmarks:
            for landmark in hand_landmarks:
                cx, cy = int(landmark.x * w), int(landmark.y * h)
                cv2.circle(frame, (cx, cy), 5, (0, 255, 0), -1)
    
    cv2.imshow('Hand Tracking Test', frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
print('Webcam closed.')

✅ Webcam opened! Press "q" in the video window to quit.
Webcam closed.


In [10]:
import csv

# Gesture labels mapped to keyboard keys
GESTURES = {
    '1': 'thumbs_up',
    '2': 'thumbs_down',
    '3': 'peace',
    '4': 'ok',
    '5': 'open_palm',
    '6': 'fist',
    '7': 'love',
    '8': 'point_up'
}

DATA_FILE = 'gesture_data.csv'

# Create the CSV file with headers if it doesn't exist yet
if not os.path.exists(DATA_FILE):
    headers = []
    for i in range(21):  # 21 landmarks
        headers += [f'x{i}', f'y{i}', f'z{i}']
    headers.append('label')
    
    with open(DATA_FILE, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(headers)
    
    print(f'✅ Created {DATA_FILE} with {len(headers)} columns')
else:
    print(f'{DATA_FILE} already exists — will append new data to it')

✅ Created gesture_data.csv with 64 columns


In [14]:
cap = cv2.VideoCapture(0)
frame_timestamp = 0
current_gesture = None
sample_count = 0

print('=== Gesture Data Collection ===')
print('Press and HOLD a number key (1-8) to record that gesture')
print('Release the key to stop recording')
print('Press "q" to quit\n')
for key, gesture in GESTURES.items():
    print(f'{key} = {gesture}')

with open(DATA_FILE, 'a', newline='') as f:
    writer = csv.writer(f)
    
    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            break
        
        frame = cv2.flip(frame, 1)
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
        
        frame_timestamp = int(time.time() * 1000)
        result = detector.detect_for_video(mp_image, frame_timestamp)
        
        key_pressed = cv2.waitKey(1) & 0xFF
        
        # Check if a valid gesture key is being held
        key_char = chr(key_pressed) if key_pressed != 255 else None
        
        if key_char in GESTURES:
            current_gesture = GESTURES[key_char]
        elif key_pressed == 255:  # no key held
            current_gesture = None
        
        if result.hand_landmarks:
            h, w, _ = frame.shape
            for hand_landmarks in result.hand_landmarks:
                # Draw the landmarks
                for landmark in hand_landmarks:
                    cx, cy = int(landmark.x * w), int(landmark.y * h)
                    cv2.circle(frame, (cx, cy), 4, (0, 255, 0), -1)
                
                # If a gesture key is currently held, save this frame's data
                if current_gesture:
                    row = []
                    for landmark in hand_landmarks:
                        row += [landmark.x, landmark.y, landmark.z]
                    row.append(current_gesture)
                    writer.writerow(row)
                    sample_count += 1
        
        # Display status on screen
        status_text = f'Recording: {current_gesture}' if current_gesture else 'Press a number key...'
        color = (0, 0, 255) if current_gesture else (255, 255, 255)
        cv2.putText(frame, status_text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)
        cv2.putText(frame, f'Total samples: {sample_count}', (10, 70), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
        
        cv2.imshow('Gesture Data Collection', frame)
        
        if key_pressed == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()
print(f'\n✅ Collection session ended. Total samples recorded: {sample_count}')

=== Gesture Data Collection ===
Press and HOLD a number key (1-8) to record that gesture
Release the key to stop recording
Press "q" to quit

1 = thumbs_up
2 = thumbs_down
3 = peace
4 = ok
5 = open_palm
6 = fist
7 = love
8 = point_up

✅ Collection session ended. Total samples recorded: 1294


In [15]:
import pandas as pd

data = pd.read_csv(DATA_FILE)
print(f'Total samples: {len(data)}')
print(f'\nSamples per gesture:')
print(data['label'].value_counts())

Total samples: 1294

Samples per gesture:
label
thumbs_up      260
open_palm      163
love           162
ok             156
fist           150
thumbs_down    142
point_up       140
peace          121
Name: count, dtype: int64


In [16]:
print(data.head(3))
print(f'\nColumns: {list(data.columns)}')
print(f'\nNumber of feature columns: {len(data.columns) - 1}')  

         x0        y0            z0        x1        y1        z1        x2  \
0  0.758882  0.858773 -6.695479e-07  0.701005  0.734909 -0.009751  0.674792   
1  0.770464  0.786026 -4.604918e-07  0.748301  0.680290  0.000579  0.696737   
2  0.771694  0.787843 -4.727471e-07  0.748956  0.683113  0.001205  0.696987   

         y2        z2        x3  ...       x18       y18       z18       x19  \
0  0.610809 -0.024206  0.665939  ...  0.649479  0.792929 -0.104480  0.662219   
1  0.573164 -0.009194  0.657130  ...  0.592669  0.784708 -0.094933  0.621080   
2  0.574946 -0.009114  0.657757  ...  0.591915  0.782072 -0.097051  0.621522   

        y19       z19       x20       y20       z20      label  
0  0.811682 -0.096411  0.694520  0.808030 -0.089203  thumbs_up  
1  0.801609 -0.085960  0.650130  0.794715 -0.077156  thumbs_up  
2  0.797684 -0.087800  0.652112  0.790714 -0.078486  thumbs_up  

[3 rows x 64 columns]

Columns: ['x0', 'y0', 'z0', 'x1', 'y1', 'z1', 'x2', 'y2', 'z2', 'x3', 'y3', 'z

In [17]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Separate features and target
X = data.drop(columns=['label'])
y = data['label']

# Encode labels (text -> numbers)
le_gesture = LabelEncoder()
y_encoded = le_gesture.fit_transform(y)

print(f'Classes: {list(le_gesture.classes_)}')
print(f'Feature shape: {X.shape}')

Classes: ['fist', 'love', 'ok', 'open_palm', 'peace', 'point_up', 'thumbs_down', 'thumbs_up']
Feature shape: (1294, 63)


In [18]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f'Training set: {X_train.shape}')
print(f'Test set: {X_test.shape}')

Training set: (1035, 63)
Test set: (259, 63)


In [19]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, random_state=42),
}

results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    accuracy = accuracy_score(y_test, preds)
    results[name] = accuracy
    print(f'{name:<25} Accuracy: {accuracy*100:.2f}%')

Logistic Regression       Accuracy: 100.00%
Random Forest             Accuracy: 100.00%
Gradient Boosting         Accuracy: 100.00%


In [20]:
# Check similarity between consecutive rows of the same gesture
sample_check = data[data['label'] == 'thumbs_up'].iloc[0:3, :5]
print(sample_check)

         x0        y0            z0        x1        y1
0  0.758882  0.858773 -6.695479e-07  0.701005  0.734909
1  0.770464  0.786026 -4.604918e-07  0.748301  0.680290
2  0.771694  0.787843 -4.727471e-07  0.748956  0.683113


In [21]:
# Split each gesture chronologically (first 80% train, last 20% test)
# rather than randomly shuffling, to avoid near-duplicate leakage

train_rows = []
test_rows = []

for gesture in data['label'].unique():
    gesture_data = data[data['label'] == gesture].reset_index(drop=True)
    split_point = int(len(gesture_data) * 0.8)
    
    train_rows.append(gesture_data.iloc[:split_point])
    test_rows.append(gesture_data.iloc[split_point:])

train_data = pd.concat(train_rows, ignore_index=True)
test_data = pd.concat(test_rows, ignore_index=True)

X_train = train_data.drop(columns=['label'])
y_train = le_gesture.transform(train_data['label'])
X_test = test_data.drop(columns=['label'])
y_test = le_gesture.transform(test_data['label'])

print(f'Training set: {X_train.shape}')
print(f'Test set: {X_test.shape}')
print(f'\nTest set breakdown:')
print(test_data['label'].value_counts())

Training set: (1032, 63)
Test set: (262, 63)

Test set breakdown:
label
thumbs_up      52
open_palm      33
love           33
ok             32
fist           30
thumbs_down    29
point_up       28
peace          25
Name: count, dtype: int64


In [22]:
results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    accuracy = accuracy_score(y_test, preds)
    results[name] = accuracy
    print(f'{name:<25} Accuracy: {accuracy*100:.2f}%')

Logistic Regression       Accuracy: 100.00%
Random Forest             Accuracy: 94.27%
Gradient Boosting         Accuracy: 96.56%


In [23]:
print(f'Total rows: {len(data)}')
print(f'Exact duplicate rows: {data.duplicated().sum()}')

Total rows: 1294
Exact duplicate rows: 0


In [24]:
from sklearn.metrics import classification_report

rf_model = models['Random Forest']
rf_preds = rf_model.predict(X_test)

print(classification_report(
    y_test, rf_preds,
    target_names=le_gesture.classes_
))

              precision    recall  f1-score   support

        fist       1.00      1.00      1.00        30
        love       0.79      1.00      0.88        33
          ok       1.00      1.00      1.00        32
   open_palm       1.00      0.55      0.71        33
       peace       1.00      1.00      1.00        25
    point_up       1.00      1.00      1.00        28
 thumbs_down       0.83      1.00      0.91        29
   thumbs_up       1.00      1.00      1.00        52

    accuracy                           0.94       262
   macro avg       0.95      0.94      0.94       262
weighted avg       0.95      0.94      0.94       262



In [25]:
# Find exactly which test samples were misclassified
misclassified = X_test.copy()
misclassified['true_label'] = le_gesture.inverse_transform(y_test)
misclassified['predicted_label'] = le_gesture.inverse_transform(rf_preds)
misclassified['correct'] = misclassified['true_label'] == misclassified['predicted_label']

errors = misclassified[~misclassified['correct']]
print(f'Total errors: {len(errors)}')
print(f'\nError breakdown (true -> predicted):')
print(errors.groupby(['true_label', 'predicted_label']).size())

Total errors: 15

Error breakdown (true -> predicted):
true_label  predicted_label
open_palm   love               9
            thumbs_down        6
dtype: int64


In [26]:
## 🔍 Error Analysis

Random Forest achieved 94.27% accuracy on a properly time-ordered 
(non-leaked) train/test split. All 15 errors in the 262-sample 
test set were concentrated in a single, well-isolated confusion:

| True Label | Predicted As | Count |
|---|---|---|
| open_palm | love | 9 |
| open_palm | thumbs_down | 6 |

No other gesture pair was ever confused — this is a specific, 
narrow failure mode rather than general model weakness.

### Likely Cause
open_palm relies on all 5 fingers being visibly extended. If the 
hand was tilted relative to the camera during recording, MediaPipe 
may have seen some fingers as foreshortened or partially occluded 
from that single 2D viewpoint — producing landmark coordinates that 
resemble curled fingers, the defining feature separating love and 
thumbs_down from an open palm.

### Note on Model Comparison
Logistic Regression reached 100% and Gradient Boosting reached 
96.56% on this same split. This doesn't necessarily mean Logistic 
Regression is the "best" model — with only 8 gestures that are 
largely distinct in landmark space, this dataset may not be 
challenging enough to meaningfully separate model quality. A future 
iteration with more, and more visually-similar, gestures (e.g. 
full ASL alphabet) would be a better stress test for model choice.

### Next Steps
- Collect additional open_palm samples specifically at varied 
  hand angles relative to camera
- Test with a larger, more ambiguous gesture set to properly 
  compare model performance

SyntaxError: invalid character '—' (U+2014) (1328160597.py, line 12)

In [29]:
cap = cv2.VideoCapture(0)
frame_timestamp = 0

print('Live gesture prediction — press "q" to quit')

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break
    
    frame = cv2.flip(frame, 1)
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
    
    frame_timestamp = int(time.time() * 1000)
    result = detector.detect_for_video(mp_image, frame_timestamp)
    
    predicted_gesture = 'No hand detected'
    
    if result.hand_landmarks:
        h, w, _ = frame.shape
        for hand_landmarks in result.hand_landmarks:
            for landmark in hand_landmarks:
                cx, cy = int(landmark.x * w), int(landmark.y * h)
                cv2.circle(frame, (cx, cy), 4, (0, 255, 0), -1)
            
            # Build the row in EXACTLY the same order as training data
            row = []
            for landmark in hand_landmarks:
                row += [landmark.x, landmark.y, landmark.z]
            
            row_df = pd.DataFrame([row], columns=X.columns)
            prediction = rf_model.predict(row_df)
            predicted_gesture = le_gesture.inverse_transform(prediction)[0]
    
    cv2.putText(frame, f'Gesture: {predicted_gesture}', (10, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 255), 2)
    
    cv2.imshow('Live Gesture Prediction', frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
print('Done!')

Live gesture prediction — press "q" to quit
Done!


In [30]:
## 🔍 Extended Error Analysis (Live Testing)

Live webcam testing revealed a broader confusion pattern than the 
formal test set alone showed. The test set (262 samples) only 
surfaced open_palm confusion, but live use surfaced two additional 
pairs:

- open_palm ↔ thumbs_down / love (confirmed in test set: 15 errors)
- ok ↔ thumbs_down (angle-dependent, live only)
- love ↔ thumbs_down (angle-dependent, live only)

**Pattern:** thumbs_down appears in every confusion pair found, 
both in formal testing and live use. Its landmark signature — 
thumb extended, four fingers curled — sits geometrically close to 
several other gestures depending on hand angle, making it a 
recurring point of ambiguity across the gesture set.

**Implication:** the 262-sample test set, while properly split to 
avoid leakage, doesn't fully capture the range of hand angles a 
user naturally produces during live, spontaneous use versus a 
single controlled recording session. This is a known limitation of 
small, single-session datasets.

### Next Steps
- Collect a second recording session for thumbs_down specifically, 
  at deliberately varied angles, to give the model more coverage 
  of its true shape variation
- Consider whether thumbs_down is worth replacing with a more 
  visually distinct gesture in a future iteration

SyntaxError: invalid character '↔' (U+2194) (1090852257.py, line 8)

In [31]:
print(f'CSV file exists: {os.path.exists(DATA_FILE)}')
print(f'Current total samples: {len(pd.read_csv(DATA_FILE))}')
print(f'Detector object still active: {detector is not None}')

CSV file exists: True
Current total samples: 1294
Detector object still active: True


In [33]:
cap = cv2.VideoCapture(0)
frame_timestamp = 0
current_gesture = None
sample_count = 0

print('=== Additional Thumbs Down Collection (Varied Angles) ===')
print('Press and HOLD key "2" for thumbs_down')
print('Vary your hand angle while holding - tilt, rotate, move closer/further')
print('Press "q" to quit\n')

with open(DATA_FILE, 'a', newline='') as f:
    writer = csv.writer(f)
    
    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            break
        
        frame = cv2.flip(frame, 1)
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
        
        frame_timestamp = int(time.time() * 1000)
        result = detector.detect_for_video(mp_image, frame_timestamp)
        
        key_pressed = cv2.waitKey(1) & 0xFF
        key_char = chr(key_pressed) if key_pressed != 255 else None
        
        if key_char == '2':
            current_gesture = 'thumbs_down'
        elif key_pressed == 255:
            current_gesture = None
        
        if result.hand_landmarks:
            h, w, _ = frame.shape
            for hand_landmarks in result.hand_landmarks:
                for landmark in hand_landmarks:
                    cx, cy = int(landmark.x * w), int(landmark.y * h)
                    cv2.circle(frame, (cx, cy), 4, (0, 255, 0), -1)
                
                if current_gesture:
                    row = []
                    for landmark in hand_landmarks:
                        row += [landmark.x, landmark.y, landmark.z]
                    row.append(current_gesture)
                    writer.writerow(row)
                    sample_count += 1
        
        status_text = f'Recording: {current_gesture}' if current_gesture else 'Hold key "2" for thumbs_down...'
        color = (0, 0, 255) if current_gesture else (255, 255, 255)
        cv2.putText(frame, status_text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)
        cv2.putText(frame, f'New samples: {sample_count}', (10, 70), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
        
        cv2.imshow('Additional Data Collection', frame)
        
        if key_pressed == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()
print(f'\n✅ Additional samples recorded: {sample_count}')

=== Additional Thumbs Down Collection (Varied Angles) ===
Press and HOLD key "2" for thumbs_down
Vary your hand angle while holding - tilt, rotate, move closer/further
Press "q" to quit


✅ Additional samples recorded: 289


In [34]:
data = pd.read_csv(DATA_FILE)
print(f'Total samples now: {len(data)}')
print(data['label'].value_counts())

Total samples now: 1583
label
thumbs_down    431
thumbs_up      260
open_palm      163
love           162
ok             156
fist           150
point_up       140
peace          121
Name: count, dtype: int64


In [35]:
# Rebuild the chronological split with updated data
train_rows = []
test_rows = []

for gesture in data['label'].unique():
    gesture_data = data[data['label'] == gesture].reset_index(drop=True)
    split_point = int(len(gesture_data) * 0.8)
    
    train_rows.append(gesture_data.iloc[:split_point])
    test_rows.append(gesture_data.iloc[split_point:])

train_data = pd.concat(train_rows, ignore_index=True)
test_data = pd.concat(test_rows, ignore_index=True)

X_train = train_data.drop(columns=['label'])
y_train = le_gesture.transform(train_data['label'])
X_test = test_data.drop(columns=['label'])
y_test = le_gesture.transform(test_data['label'])

print(f'Training set: {X_train.shape}')
print(f'Test set: {X_test.shape}')

# Retrain Random Forest
rf_model = RandomForestClassifier(n_estimators=200, random_state=42)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)

accuracy = accuracy_score(y_test, rf_preds)
print(f'\nNew accuracy: {accuracy*100:.2f}%')
print(f'\n{classification_report(y_test, rf_preds, target_names=le_gesture.classes_)}')

Training set: (1263, 63)
Test set: (320, 63)

New accuracy: 95.31%

              precision    recall  f1-score   support

        fist       1.00      1.00      1.00        30
        love       0.69      1.00      0.81        33
          ok       1.00      1.00      1.00        32
   open_palm       1.00      0.55      0.71        33
       peace       1.00      1.00      1.00        25
    point_up       1.00      1.00      1.00        28
 thumbs_down       1.00      1.00      1.00        87
   thumbs_up       1.00      1.00      1.00        52

    accuracy                           0.95       320
   macro avg       0.96      0.94      0.94       320
weighted avg       0.97      0.95      0.95       320



In [36]:
misclassified = X_test.copy()
misclassified['true_label'] = le_gesture.inverse_transform(y_test)
misclassified['predicted_label'] = le_gesture.inverse_transform(rf_preds)
misclassified['correct'] = misclassified['true_label'] == misclassified['predicted_label']

errors = misclassified[~misclassified['correct']]
print(f'Total errors: {len(errors)}')
print(f'\nError breakdown (true -> predicted):')
print(errors.groupby(['true_label', 'predicted_label']).size())

Total errors: 15

Error breakdown (true -> predicted):
true_label  predicted_label
open_palm   love               15
dtype: int64


In [37]:
cap = cv2.VideoCapture(0)
frame_timestamp = 0
current_gesture = None
sample_count = 0

print('=== Additional Open Palm Collection (Varied Angles) ===')
print('Press and HOLD key "5" for open_palm')
print('Vary your hand angle while holding - tilt, rotate, move closer/further')
print('Press "q" to quit\n')

with open(DATA_FILE, 'a', newline='') as f:
    writer = csv.writer(f)
    
    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            break
        
        frame = cv2.flip(frame, 1)
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
        
        frame_timestamp = int(time.time() * 1000)
        result = detector.detect_for_video(mp_image, frame_timestamp)
        
        key_pressed = cv2.waitKey(1) & 0xFF
        key_char = chr(key_pressed) if key_pressed != 255 else None
        
        if key_char == '5':
            current_gesture = 'open_palm'
        elif key_pressed == 255:
            current_gesture = None
        
        if result.hand_landmarks:
            h, w, _ = frame.shape
            for hand_landmarks in result.hand_landmarks:
                for landmark in hand_landmarks:
                    cx, cy = int(landmark.x * w), int(landmark.y * h)
                    cv2.circle(frame, (cx, cy), 4, (0, 255, 0), -1)
                
                if current_gesture:
                    row = []
                    for landmark in hand_landmarks:
                        row += [landmark.x, landmark.y, landmark.z]
                    row.append(current_gesture)
                    writer.writerow(row)
                    sample_count += 1
        
        status_text = f'Recording: {current_gesture}' if current_gesture else 'Hold key "5" for open_palm...'
        color = (0, 0, 255) if current_gesture else (255, 255, 255)
        cv2.putText(frame, status_text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)
        cv2.putText(frame, f'New samples: {sample_count}', (10, 70), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
        
        cv2.imshow('Additional Data Collection', frame)
        
        if key_pressed == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()
print(f'\n✅ Additional samples recorded: {sample_count}')

=== Additional Open Palm Collection (Varied Angles) ===
Press and HOLD key "5" for open_palm
Vary your hand angle while holding - tilt, rotate, move closer/further
Press "q" to quit


✅ Additional samples recorded: 280


In [38]:
data = pd.read_csv(DATA_FILE)
print(f'Total samples: {len(data)}')
print(data['label'].value_counts())

train_rows = []
test_rows = []

for gesture in data['label'].unique():
    gesture_data = data[data['label'] == gesture].reset_index(drop=True)
    split_point = int(len(gesture_data) * 0.8)
    
    train_rows.append(gesture_data.iloc[:split_point])
    test_rows.append(gesture_data.iloc[split_point:])

train_data = pd.concat(train_rows, ignore_index=True)
test_data = pd.concat(test_rows, ignore_index=True)

X_train = train_data.drop(columns=['label'])
y_train = le_gesture.transform(train_data['label'])
X_test = test_data.drop(columns=['label'])
y_test = le_gesture.transform(test_data['label'])

rf_model = RandomForestClassifier(n_estimators=200, random_state=42)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)

accuracy = accuracy_score(y_test, rf_preds)
print(f'\nNew accuracy: {accuracy*100:.2f}%')
print(f'\n{classification_report(y_test, rf_preds, target_names=le_gesture.classes_)}')

Total samples: 1863
label
open_palm      443
thumbs_down    431
thumbs_up      260
love           162
ok             156
fist           150
point_up       140
peace          121
Name: count, dtype: int64

New accuracy: 99.47%

              precision    recall  f1-score   support

        fist       1.00      1.00      1.00        30
        love       1.00      1.00      1.00        33
          ok       1.00      1.00      1.00        32
   open_palm       1.00      1.00      1.00        89
       peace       1.00      1.00      1.00        25
    point_up       0.93      1.00      0.97        28
 thumbs_down       1.00      1.00      1.00        87
   thumbs_up       1.00      0.96      0.98        52

    accuracy                           0.99       376
   macro avg       0.99      1.00      0.99       376
weighted avg       1.00      0.99      0.99       376



In [39]:
misclassified = X_test.copy()
misclassified['true_label'] = le_gesture.inverse_transform(y_test)
misclassified['predicted_label'] = le_gesture.inverse_transform(rf_preds)
misclassified['correct'] = misclassified['true_label'] == misclassified['predicted_label']

errors = misclassified[~misclassified['correct']]
print(f'Total errors: {len(errors)}')
print(errors.groupby(['true_label', 'predicted_label']).size())

Total errors: 2
true_label  predicted_label
thumbs_up   point_up           2
dtype: int64
